In [1]:
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
import pickle

from data.hiv_simulator import HIVSimulator

# Load Dataset

In [2]:
all_episodes = {}
all_episodes[0.5] = pickle.load(open('data/batch_trajectories_epsilon=05.p', 'rb'))
all_episodes[0.2] = pickle.load(open('data/batch_trajectories_epsilon=02.p', 'rb'))

In [3]:
print("Number of trajectories/episodes", len(all_episodes[0.2]))
print("Each episode comprises of {} lists. All lists are the same length. These represent (S, A, R, S', p) tuples".format(len(all_episodes[0.2][0])) )

Number of trajectories/episodes 250
Each episode comprises of 5 lists. All lists are the same length. These represent (S, A, R, S', p) tuples


In [4]:
# Examine the first episode
first_episode = all_episodes[0.2][0]

states = first_episode[0]
actions = first_episode[1]
rewards = first_episode[2]
next_states = first_episode[3]
propensities = first_episode[4] # probability of the selected action under the behavior policy

In [5]:
print( np.array(states[0:5]) )

[[5.21371162 0.69897    4.07718615 1.66275783 4.80562997 1.38021124]
 [5.30317939 1.74918268 2.93439581 1.43190928 3.53918668 1.4207034 ]
 [5.37917686 2.15822386 2.09293134 1.19502454 2.88451327 1.54718007]
 [5.43770298 2.05278582 3.06321797 1.87146423 3.79844829 1.65002342]
 [5.48880868 2.1022221  2.44742931 1.49258374 3.22973987 1.73588493]]


In [6]:
print( np.array(actions[0:5]) )

[3 1 0 1 3]


In [7]:
print( np.array(rewards[0:5]) )

[1.60192273 2.53750502 4.4042061  4.44661169 5.33127257]


In [8]:
print( np.array(next_states[0:5]) )

[[5.30317939 1.74918268 2.93439581 1.43190928 3.53918668 1.4207034 ]
 [5.37917686 2.15822386 2.09293134 1.19502454 2.88451327 1.54718007]
 [5.43770298 2.05278582 3.06321797 1.87146423 3.79844829 1.65002342]
 [5.48880868 2.1022221  2.44742931 1.49258374 3.22973987 1.73588493]
 [5.53366048 2.3834204  1.67247247 0.91825541 2.32646703 1.80149929]]


In [9]:
print( np.array(propensities[0:5]) )

[0.85 0.05 0.85 0.85 0.85]


# Solutions

In [10]:
H = 5 # horizon

# (1a)

In [11]:
# Load relevant episodes
episodes = all_episodes[0.5]
T = len(episodes) # number of episodes

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute importance sampling estimator
G = np.array(G)
W = np.array(W)
IS_estimate = np.mean(G * W)

# Confidence interval
std_error = np.sqrt(np.var(G * W) / T)

print(f'Importance Sampling Estimate for π_u (ϵ=0.5): {IS_estimate}')
print(f'95% Confidence Interval: [{IS_estimate - 1.96 * std_error}, {IS_estimate + 1.96 * std_error}]')

Importance Sampling Estimate for π_u (ϵ=0.5): 16.022712847503588
95% Confidence Interval: [10.240930904386314, 21.80449479062086]


# (1b)

In [12]:
# Load relevant episodes
episodes = all_episodes[0.2]
T = len(episodes) # number of episodes

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute importance sampling estimator
G = np.array(G)
W = np.array(W)
IS_estimate = np.mean(G * W)

# Confidence interval
std_error = np.sqrt(np.var(G * W) / T)

print(f'Importance Sampling Estimate for π_u (ϵ=0.2): {IS_estimate}')
print(f'95% Confidence Interval: [{IS_estimate - 1.96 * std_error}, {IS_estimate + 1.96 * std_error}]')

Importance Sampling Estimate for π_u (ϵ=0.2): 17.487166243967142
95% Confidence Interval: [-8.804967590971582, 43.77930007890586]


# (1c)

In [13]:
def compute_ess(W):
    """Compute the ESS"""
    W_sum = np.sum(W)
    W_squared_sum = np.sum(W**2)
    ess = (W_sum**2) / W_squared_sum
    return ess

In [14]:
# Compute ESS for ϵ=0.5

# Load relevant episodes
episodes = all_episodes[0.5]

# Initialise list to store results
W = []  # list of importance sampling ratios

for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    W.append(importance_sampling_ratio)

W = np.array(W)
ess = compute_ess(W)
print(f'Effective Sample Size for π_u (ϵ=0.5): {ess}')

Effective Sample Size for π_u (ϵ=0.5): 23.615677214314246


In [15]:
# Compute ESS for ϵ=0.2

# Load relevant episodes
episodes = all_episodes[0.2]

# Initialise list to store results
W = []  # list of importance sampling ratios

for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    W.append(importance_sampling_ratio)

W = np.array(W)
ess = compute_ess(W)
print(f'Effective Sample Size for π_u (ϵ=0.2): {ess}')

Effective Sample Size for π_u (ϵ=0.2): 1.7810722282019509


# (1d)

In [16]:
# Load relevant episodes
episodes = all_episodes[0.5]

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute weighted importance sampling estimator
G = np.array(G)
W = np.array(W)
WIS_estimate = np.sum(G * W) / np.sum(W)

print(f'Weighted Importance Sampling Estimate for π_u (ϵ=0.5): {WIS_estimate}')

Weighted Importance Sampling Estimate for π_u (ϵ=0.5): 13.198579960810633


# (2b)

In [ ]:
class FittedQ(object):
    def __init__(self, regressor=None):
        """Initialize simulator and regressor. Can optionally pass a custom
        `regressor` model (which must implement `fit` and `predict` -- you can
        use this to try different models like linear regression or NNs)"""
        self.simulator = HIVSimulator()
        self.regressor = regressor or ExtraTreesRegressor(n_estimators=10)
        self.action_codes = np.array(self.simulator.binary_action_codes)


    def Q(self, states):
        """Return the Q function estimate of `states` for each action"""
        # Uuse the trained regression model
        if len(states.shape) == 1:
            stateactions = np.concatenate( [np.tile(states,(4,1)), self.action_codes], axis=1 )
            rewards = self.regressor.predict( stateactions )
            return np.expand_dims(rewards,0)

        all_rewards = []
        for action in self.action_codes:
            stateactions = np.concatenate( [states, np.tile(action, (len(states), 1) )], axis=1 )
            rewards = self.regressor.predict( stateactions )
            all_rewards.append( rewards )
            
        return np.transpose( np.vstack( all_rewards ) )

regressor = pickle.load( open('data/fitted_Q_regressor.p', 'rb') )
Q_function = FittedQ(regressor)

In [18]:
def pi_q(state):
    """Return the selected action under π_q based on the Q-function"""
    q_values = Q_function.Q(state)[0]
    return np.argmax(q_values)

In [19]:
# Load relevant episodes
episodes = all_episodes[0.2]
T = len(episodes) # number of episodes

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = 1.0
    for h in range(H):
        pi_q_propensity = 1.0 if actions[h] == pi_q(states[h]) else 0.0
        importance_sampling_ratio *= pi_q_propensity / propensities[h]
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute importance sampling estimator
G = np.array(G)
W = np.array(W)
IS_estimate = np.mean(G * W)

# Confidence interval
std_error = np.sqrt(np.var(G * W) / T)

print(f'Importance Sampling Estimate for π_q (ϵ=0.2): {IS_estimate}')
print(f'95% Confidence Interval: [{IS_estimate - 1.96 * std_error}, {IS_estimate + 1.96 * std_error}]')

Importance Sampling Estimate for π_q (ϵ=0.2): 20.761611815698778
95% Confidence Interval: [18.03958063535283, 23.483642996044725]
